# Day 01 · RAG QA 봇 실습 (로컬)
임베딩·벡터DB는 **로컬(CPU)**, 생성은 **NVIDIA build**(Qwen 모델, 개인 토큰). 7단계 RAG를 직접 만듭니다.

`적재 → 청킹 → 임베딩 → 저장 → 검색 → 증강 → 생성`

## 0 · 설치


In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai


## 1 · NVIDIA API 토큰 적용 🔑
생성(LLM)은 **NVIDIA build** 의 Qwen 모델을 씁니다. 개인 **NVIDIA API 토큰**(`nvapi-...`)이 필요해요.

- 토큰이 아직 없다면 → **`실습_시작하기.ipynb`** 에서 발급 (build.nvidia.com · 무료 · 카드 불필요)
- 아래 셀을 실행하면 토큰을 **직접 입력**받습니다(화면에 안 보임). 이미 `NVIDIA_API_KEY` 환경변수가 있으면 자동 사용.

> 임베딩·벡터DB는 로컬(CPU)이라 토큰이 필요 없고, **생성만** 토큰을 씁니다.

In [ ]:
import os, getpass
from openai import OpenAI

# 생성 엔드포인트: NVIDIA build (기본). 강사 DGX로 바꾸려면 아래 줄 주석 해제.
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
# LLM_BASE_URL = "http://124.51.229.210:30001/v1"   # ← 강사 DGX 백업

# NVIDIA API 토큰: 환경변수(NVIDIA_API_KEY) 있으면 자동, 없으면 직접 입력(화면 비표시)
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")

# 사용 가능한 모델에서 Qwen 채팅 모델 자동 확정
_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"   # 로컬 다국어 임베딩
QDRANT_PATH = "./qdrant_db"

## 2 · 문서 적재 (예시 코퍼스)
실제로는 사내 문서를 넣습니다. 여기선 작은 예시로 흐름을 익힙니다.


In [ ]:
docs = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받아야 한다.",
    "비용 정산은 영수증을 첨부해 신청서를 작성하고, 경영지원팀 검토 후 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며, 전날 18시까지 팀 채널에 사전 공유한다.",
    "보안 정책상 외부 USB 사용은 금지되며, 파일 공유는 사내 드라이브를 사용한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며, 재로그인으로 해결된다.",
]


## 3 · 청킹
문장이 짧아 그대로 두지만, 긴 문서는 RecursiveCharacterTextSplitter로 분할합니다.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = []
for d in docs:
    chunks.extend(splitter.split_text(d))
print(len(chunks), "청크")
chunks[:3]


## 4 · 임베딩 (로컬·CPU)
정규화하면 코사인=내적이라 검색이 간단해집니다.


In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)

def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True)

vecs = embed(chunks)
print("임베딩 shape:", vecs.shape)   # (청크 수, 차원)


## 5 · 벡터 DB 저장 (로컬 Qdrant)


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(path=QDRANT_PATH)   # 로컬 파일 모드
dim = vecs.shape[1]
client.recreate_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)
client.upsert("docs", points=[
    PointStruct(id=i, vector=v.tolist(), payload={"text": chunks[i]})
    for i, v in enumerate(vecs)
])
print("저장 완료:", client.count("docs").count, "청크")


## 6 · 검색 (Top-K)
질문을 임베딩해 가장 가까운 청크를 찾습니다. (`query_points` 사용)


In [ ]:
def search(query, k=3):
    qv = embed([query])[0]
    res = client.query_points("docs", query=qv.tolist(), limit=k).points
    return [(p.payload["text"], round(p.score, 3)) for p in res]

for t, s in search("연차 며칠 전에 신청해요?"):
    print(round(s,3) if isinstance(s,float) else s, "|", t)


## 7 · 증강 + 생성 (NVIDIA · Qwen)
검색된 근거만 사용하고, 없으면 '모른다'고 답하도록 프롬프트를 단단히 만듭니다.

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)   # NVIDIA build (Qwen)

SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 자료에 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (근거: 자료번호)를 표기하라. 추측 금지.")

def rag_answer(query, k=3):
    hits = search(query, k)
    context = "\n".join(f"[{i+1}] {t}" for i,(t,_) in enumerate(hits))
    msg = [{"role":"system","content":SYSTEM},
           {"role":"user","content":f"[자료]\n{context}\n\n[질문] {query}"}]
    out = llm.chat.completions.create(model=LLM_MODEL, messages=msg, temperature=0.2)
    return out.choices[0].message.content

print(rag_answer("연차 휴가는 며칠 전에 신청하나요?"))

## 8 · 테스트 & 환각 점검
문서에 **없는** 질문에 '문서에 없음'으로 답하는지 확인하세요.


In [ ]:
for q in ["비용 정산은 누가 최종 승인하나요?",
          "ERR-4021은 무슨 뜻인가요?",
          "사내 헬스장 운영시간은?"]:   # 마지막은 문서에 없음 → '문서에 없음' 기대
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-"*50)


## 산출물 체크리스트
- ✅ 로컬 임베딩 + 로컬 Qdrant + **NVIDIA(Qwen) 생성**으로 RAG 동작
- ✅ 검색된 근거로만 답하고, 없으면 '문서에 없음'
- ✅ 내 문서로 `docs`만 바꾸면 나만의 QA 봇 완성